In [0]:
from pyspark.sql.types import *

claims_schema = StructType([
    StructField("ClaimID", LongType(), True),
    StructField("MemberID", StringType(), True),
    StructField("ProviderID", StringType(), True),
    StructField("Hospital", StringType(), True),
    StructField("ClaimAmount", LongType(), True),
    StructField("ClaimDate", DateType(), True),
    StructField("Status", StringType(), True)
])

In [0]:
source_path = "abfss://source@mvnazureproject.dfs.core.windows.net/claims/"

schema_path = "abfss://bronze@mvnazureproject.dfs.core.windows.net/schema/health_claims"

checkpoint_path = "abfss://bronze@mvnazureproject.dfs.core.windows.net/checkpoints/health_claims"

bronze_table = "mvn_db_catalog.bronze.health_claims"

In [0]:
df = (
    spark.readStream
         .format("cloudFiles")
         .option("cloudFiles.format", "csv")
         .option("header", "true")
         .option("cloudFiles.schemaLocation", schema_path)
         .schema(claims_schema)
         .load(source_path)
)

In [0]:
df.printSchema()

print(df.isStreaming)


In [0]:
query = (
    df.writeStream
      .format("delta")
      .option("checkpointLocation", checkpoint_path)
      .trigger(availableNow=True)
      .toTable(bronze_table)
)

In [0]:

display(
    df,
    checkpointLocation="abfss://bronze@mvnazureproject.dfs.core.windows.net/checkpoints/display_health_claims"
)